# 00. SurvFace Fine-Tuning 및 1:N 보정 프로토콜 분할 가이드

본 노트북은 UCFace 논문(IEEE T-IFS 2024, DOI: [10.1109/TIFS.2024.3426973](https://doi.org/10.1109/TIFS.2024.3426973))의 SurvFace 미세조정(Fine-Tuning) 프로토콜을 구현하고, 기존 1:N 압축 검색 및 연속 신뢰도 보정(Continuous FIQA Calibration) 파이프라인 간의 **인물 누출(Identity Leakage)을 100% 방지**하기 위한 4-Role Identity 분할 매니페스트(`training_manifest_v2_finetune.csv`)를 생성합니다.

### 참고 문헌
1. **UCFace**: Z. Shen et al., *Uncertainty-Aware Contrastive Face Recognition*, IEEE T-IFS 2024.
2. **QMUL-SurvFace**: L. Cheng et al., *SurvFace: Comprehensive Benchmark on Surveillance Face Recognition*, IEEE T-IFS 2020.
3. **ArcFace**: J. Deng et al., *ArcFace: Additive Angular Margin Loss for Deep Face Recognition*, CVPR 2019.

### Identity 예산 분배 원칙 (Total: 5,319 IDs, 220,888 Images)
- **FR Fine-Tuning Train**: 1,600 인물 (MobileFaceNet 베이스라인 학습용)
- **FR Fine-Tuning Val**: 219 인물 (조기 종료 및 검증 지표 측정용)
- **1:N Calibration Gallery**: 3,000 인물 (사후 임계값 보정 갤러리/Watchlist용)
- **1:N Calibration Probes**: 500 인물 (Non-mated 사후 임계값 보정 탐색용)
- **상호 배타성(Disjointness)**: 4개 세트 간 인물 교집합은 정확히 0이며, 공식 테스트셋(Test Gallery & Probes)과는 완전히 분리됩니다.

In [ ]:
# cell 1 : 기본 라이브러리 임포트 및 환경 확인
from __future__ import annotations
from pathlib import Path
import sys
import torch
import pandas as pd

# 프로젝트 루트 경로 설정
PROJECT_ROOT = Path("../..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.training.fr_baselines.protocol import build_survface_finetune_manifest
from research.protocols.open_set import validate_identity_disjoint_splits

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA is not available. Using CPU.")

In [ ]:
# cell 2 : 파라미터 및 경로 설정
SURVFACE_ROOT = PROJECT_ROOT / "data/raw/QMUL-SurvFace"
MANIFEST_OUTPUT = PROJECT_ROOT / "data/manifests/training_manifest_v2_finetune.csv"

EXECUTE_DATA_PREP = False  # 안전 기본값: False (사용자 승인 후 True로 변경하여 생성)
KEEP_RAW_RESULTS = True   # 중간 집계 및 감사 결과 보존 여부
SEED = 8972

TRAIN_ID_COUNT = 1600
VAL_ID_COUNT = 219
CALIB_GALLERY_COUNT = 3000
CALIB_PROBE_COUNT = 500

print(f"SurvFace root: {SURVFACE_ROOT}")
print(f"Target manifest: {MANIFEST_OUTPUT}")
print(f"Split budget: Train={TRAIN_ID_COUNT}, Val={VAL_ID_COUNT}, CalibGal={CALIB_GALLERY_COUNT}, CalibPrb={CALIB_PROBE_COUNT}")

In [ ]:
# cell 3 : SurvFace 학습셋 디렉터리 감사
training_set_dir = SURVFACE_ROOT / "training_set"
if training_set_dir.is_dir():
    identities = [p for p in training_set_dir.iterdir() if p.is_dir()]
    print(f"[Audit] Found {len(identities)} identity folders in {training_set_dir}")
    sample_img_count = sum(len(list(p.glob('*.jpg'))) for p in identities[:10])
    print(f"[Audit] Sample 10 identities have {sample_img_count} images total")
else:
    print(f"[Notice] {training_set_dir} not yet mounted. Mock/Protocol test mode will be verified.")

In [ ]:
# cell 4 : 4-Role Leak-Free 프로토콜 분할 매니페스트 생성
if EXECUTE_DATA_PREP and training_set_dir.is_dir():
    bundle = build_survface_finetune_manifest(
        survface_root=SURVFACE_ROOT,
        project_root=PROJECT_ROOT,
        seed=SEED,
        train_identities_count=TRAIN_ID_COUNT,
        val_identities_count=VAL_ID_COUNT,
        calib_gallery_count=CALIB_GALLERY_COUNT,
        calib_probe_count=CALIB_PROBE_COUNT,
        output_csv_path=MANIFEST_OUTPUT,
    )
    print("[Success] Manifest generated successfully!")
    print(bundle.summary)
else:
    print("[Notice] EXECUTE_DATA_PREP=False. Manifest creation skipped to prevent unwanted file modification.")

In [ ]:
# cell 5 : 매니페스트 무결성 검증 및 역할별 통계 점검
if MANIFEST_OUTPUT.is_file():
    df = pd.read_csv(MANIFEST_OUTPUT)
    validate_identity_disjoint_splits(df)
    print(f"Total rows in manifest: {len(df)}")
    print("\n--- Role Breakdown ---")
    role_counts = df.groupby('finetune_role')['identity_id'].nunique()
    img_counts = df['finetune_role'].value_counts()
    summary_df = pd.DataFrame({'Identities': role_counts, 'Images': img_counts})
    print(summary_df)
    print("\n[Integrity Check] 100% Identity-Disjoint validated successfully!")
else:
    print(f"Manifest file {MANIFEST_OUTPUT} does not exist yet. Run Cell 4 with EXECUTE_DATA_PREP=True when ready.")